In [1]:
import numpy as np
import pandas as pd

In [2]:
def import_excel(start_year, end_year):
    df = pd.read_excel('./pop/pop_combined_2007-2019.xlsx', sheet_name=[str(i) for i in range(start_year, end_year+1)])  
    df = pd.concat(df, axis=0, ignore_index=True)
    return df

In [3]:
pop_2007_2010 = import_excel(2007, 2010)
pop_2011_2014 = import_excel(2011, 2014)
pop_2015_2019 = import_excel(2015, 2019)

In [4]:
col_mapping = {
    'Ins-\ngesamt': 'total',
    'unter 6': 'below_6',
    '6 - 15': '6-15',
    ' 6 - 15': '6-15',
    '15 - 18': '15-18',
    '18 - 27': '18-27',
    '27 - 45': '27-45',
    '45 - 55': '45-55',
    '55 - 65': '55-65',
    '65 und\nmehr': '65_above',
    'weiblich': 'female',
    'Darunter\nweiblich': 'female',
    'Jahr': 'year',
    'Bezirksregion_new': 'Bezirksregion'
}

### 2007 - 2010

In [5]:
def clean_until_2014(df):
    df['Bezirk'] = df['Bezirk'].apply('{:0>2}'.format)
    df['Bezirksregion'] = df['Bezirksregion'].apply('{:0>4}'.format)
    df['Bezirksregion_new'] = df['Bezirksregion'].apply(lambda x: x[2:])
    df['Prognoseraum'] = df['Bezirksregion'].apply(lambda x: x[:2])
    df['LOR'] = df['Bezirk'] + df['Prognoseraum'] + df['Bezirksregion_new']
    df.drop(['Bezirksregion', 'Bezirksname'], axis=1, inplace=True)
    return df

In [6]:
pop_2007_2010 = clean_until_2014(pop_2007_2010)

pop_2007_2010['below_6'] = pop_2007_2010['unter 3'] + pop_2007_2010['3 - 6']
pop_2007_2010['6-15'] = pop_2007_2010['6 - 10'] + pop_2007_2010['10 - 15']
pop_2007_2010['15-18'] = pop_2007_2010['15 - 18']
pop_2007_2010['18-27'] = pop_2007_2010['18 - 20'] + pop_2007_2010['20 - 25']
pop_2007_2010['27-45'] = pop_2007_2010['25 - 30'] + pop_2007_2010['30 - 35'] + pop_2007_2010['35 - 40'] + pop_2007_2010['40 - 45']
pop_2007_2010['45-55'] = pop_2007_2010['45 - 50'] + pop_2007_2010['50 - 55']
pop_2007_2010['55-65'] = pop_2007_2010['55 - 60'] + pop_2007_2010['60 - 63'] + pop_2007_2010['63 - 65']
pop_2007_2010['65_above'] = pop_2007_2010['65 - 70'] + pop_2007_2010['70 - 75'] + pop_2007_2010['75 - 80'] + pop_2007_2010['80 - 85'] + pop_2007_2010['85 und\nmehr']

pop_2007_2010.drop(['unter 3', '3 - 6', '6 - 10', '10 - 15', '15 - 18', '18 - 20', '20 - 25',
                    '25 - 30', '30 - 35', '35 - 40', '40 - 45', '45 - 50', '50 - 55','55 - 60',
                    '60 - 63', '63 - 65', '65 - 70', '70 - 75', '75 - 80', '80 - 85', '85 und\nmehr',
                    'Bezirk\nLOR-Bezirksreg.'], axis=1, inplace=True)

pop_2007_2010.rename(columns=col_mapping, inplace=True)

### 2011 - 2014

In [7]:
pop_2011_2014 = clean_until_2014(pop_2011_2014)
pop_2011_2014.rename(columns=col_mapping, inplace=True)

### 2015 - 2019

In [8]:
for col in ['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum']:
    pop_2015_2019[col] = pop_2015_2019[col].apply('{:0>2}'.format)

pop_2015_2019['LOR'] = pop_2015_2019['Bezirk'] + pop_2015_2019['Prognoseraum'] + pop_2015_2019['Bezirksregion']
pop_2015_2019['PR'] = pop_2015_2019['Bezirk'] + pop_2015_2019['Prognoseraum'] + pop_2015_2019['Bezirksregion'] + pop_2015_2019['Planungsraum']

pop_2015_2019.drop('Ausländer', axis=1, inplace=True)

pop_2015_2019.rename(columns=col_mapping, inplace=True)

### Adding female totals to 2007-2010 according to 2011 percentages

In [9]:
perc_f = pop_2011_2014.loc[pop_2011_2014.year==2011, ['Bezirk', 'Prognoseraum', 'Bezirksregion', 'LOR', 'female', 'total']].copy()
perc_f['female_perc'] = perc_f['female'] / perc_f['total']
perc_f.drop(['female', 'total'], axis=1, inplace=True)

In [10]:
pop_2007_2010_new = perc_f.merge(pop_2007_2010, how='outer', on=['Bezirk', 'Prognoseraum', 'Bezirksregion', 'LOR'])

In [11]:
pop_2007_2010_new['female'] = pop_2007_2010_new['female_perc'] * pop_2007_2010_new['total']
pop_2007_2010_new.drop('female_perc', axis=1, inplace=True)

### Distributing totals 2007-2014 to PR according to 2015 percentages

In [12]:
pop_2007_2014 = pd.concat([pop_2007_2010_new, pop_2011_2014], ignore_index=True)

In [13]:
cols = ['total', 'below_6', '6-15', '15-18', '18-27', '27-45', '45-55', '55-65', '65_above', 'female']
perc_age = pop_2015_2019[pop_2015_2019.year==2015].copy()

In [14]:
for col in cols:
    perc_age[f'{col}_perc'] = perc_age[col] / perc_age.groupby('LOR')[col].transform(sum)
    perc_age[f'{col}_perc'].fillna(0, inplace=True)
    perc_age.drop(col, axis=1, inplace=True)
    
perc_age.drop('year', axis=1, inplace=True)

In [15]:
pop_2007_2014_new = perc_age.merge(pop_2007_2014, how='outer', on=['Bezirk', 'Prognoseraum', 'Bezirksregion', 'LOR'])

In [16]:
for col in cols:
    pop_2007_2014_new[col] = pop_2007_2014_new[col] * pop_2007_2014_new[f'{col}_perc']
    pop_2007_2014_new.drop(f'{col}_perc', axis=1, inplace=True)

### Combining all dataframes

In [17]:
age_final = pd.concat([pop_2007_2014_new, pop_2015_2019], ignore_index=True)

### Creating age categories

In [18]:
age_final['boomers'] = age_final['55-65'] + age_final['65_above']
age_final['gen_x'] = age_final['45-55']
age_final['millenials'] = age_final['18-27'] + age_final['27-45']
age_final['children'] = age_final['below_6'] + age_final['6-15'] + age_final['15-18']

In [19]:
import joblib

joblib.dump(age_final.drop(['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum'], axis=1), './final_age.csv')

['./final_age.csv']

In [22]:
age_final.drop(['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum'], axis=1)

,LOR,PR,total,year,below_6,6-15,15-18,18-27,27-45,45-55,55-65,65_above,female,boomers,gen_x,millenials,children
0,000000,00000000,3353854.0,2007,170137.0,232813.0,86886.0,294103.0,1058262.0,491474.0,400669.0,619510.0,1.713256e+06,1020179.0,491474.0,1352365.0,489836.0
1,000000,00000000,3362842.0,2008,173830.0,233772.0,78587.0,295032.0,1046758.0,504917.0,394142.0,635804.0,1.717848e+06,1029946.0,504917.0,1341790.0,486189.0
2,000000,00000000,3369672.0,2009,178581.0,235576.0,76005.0,291531.0,1033694.0,516571.0,389595.0,648119.0,1.721337e+06,1037714.0,516571.0,1325225.0,490162.0
3,000000,00000000,3387562.0,2010,182927.0,239243.0,75001.0,287383.0,1029521.0,527782.0,395864.0,649841.0,1.730476e+06,1045705.0,527782.0,1316904.0,497171.0
4,000000,00000000,3427114.0,2011,188030.0,241745.0,76137.0,392807.0,929923.0,539668.0,407838.0,650966.0,1.750680e+06,1058804.0,539668.0,1322730.0,505912.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5788,123021,12302110,10708.0,2019,863.0,1488.0,466.0,1238.0,2485.0,1318.0,1078.0,1772.0,5.457000e+03,2850.0,1318.0,3723.0,2817.0
5789,123022,12302211,3291.0,2019,119.0,169.0,67.0,301.0,557.0,456.0,562.0,1060.0,1.668000e+03,1622.0,456.0,858.0,355.0
5790,123022,12302212,5983.0,2019,519.0,787.0,222.0,690.0,1347.0,737.0,636.0,1045.0,3.015000e+03,1681.0,737.0,2037.0,1528.0
5791,123043,12304313,6756.0,2019,346.0,390.0,119.0,709.0,1765.0,981.0,1009.0,1437.0,3.347000e+03,2446.0,981.0,2474.0,855.0
